# HFSS 2.45 GHz — Reproducible Field Analysis (Gate 1)

This Colab notebook reconstructs the complex electric field exported by HFSS and generates the **40 single-channel field-intensity maps** requested for the exploratory presentation.

**Gate 1 intentionally stops before the Shearlet transform.**

In [ ]:
# Colab environment
from pathlib import Path
import sys, os, zipfile, shutil

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)


## 1. Data source

For development, keep the HFSS database outside GitHub. Preferred option: place the original ZIP in Google Drive.

Set `DATA_SOURCE = "DRIVE"` or `"UPLOAD"` below.

In [ ]:
DATA_SOURCE = "DRIVE"  # "DRIVE" or "UPLOAD"
DATA_ZIP_NAME = "Entregas_papper(1).zip"

if IN_COLAB and DATA_SOURCE == "DRIVE":
    from google.colab import drive
    drive.mount("/content/drive")
    # CHANGE ONLY THIS PATH if needed:
    DATA_ZIP = Path("/content/drive/MyDrive/HFSS_Dataset") / DATA_ZIP_NAME
elif IN_COLAB and DATA_SOURCE == "UPLOAD":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one dataset ZIP.")
    DATA_ZIP = Path("/content") / next(iter(uploaded))
else:
    DATA_ZIP = Path("../data") / DATA_ZIP_NAME

print("Dataset ZIP:", DATA_ZIP)
if not DATA_ZIP.exists():
    raise FileNotFoundError(DATA_ZIP)


In [ ]:
# Extract once
WORK = Path("/content/hfss_work") if IN_COLAB else Path("../.work")
DATA_DIR = WORK / "dataset"
OUTPUT_DIR = WORK / "outputs"

if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_DIR)

print("Extracted dataset to:", DATA_DIR)


## 2. Gate 1 processing

The pipeline reconstructs

\[
\mathbf E = \Re\{\mathbf E\} + j\Im\{\mathbf E\}
\]

and computes

\[
|\mathbf E|=\sqrt{|E_x|^2+|E_y|^2+|E_z|^2}.
\]

The geometric mask is applied only after the full numeric field is reconstructed.

In [ ]:
# In Colab the repository should be cloned before this cell.
# Example:
# !git clone https://github.com/<USER>/<REPO>.git /content/hfss-shearlet-colab
# sys.path.insert(0, "/content/hfss-shearlet-colab")

# Local fallback when notebook lives inside this repository:
repo_candidates = [Path("/content/hfss-shearlet-colab"), Path("..").resolve()]
for candidate in repo_candidates:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Repository src/ directory not found.")

from src.pipeline import run_intensity_pipeline

rows = run_intensity_pipeline(DATA_DIR, OUTPUT_DIR)
print(f"Processed {len(rows)} cuts.")
assert len(rows) == 40, f"Expected 40 cuts, got {len(rows)}"


In [ ]:
# Gate 1 output validation
pngs = sorted((OUTPUT_DIR / "01_intensity").rglob("*.png"))
metrics = OUTPUT_DIR / "04_metrics" / "field_metrics.csv"

print("Intensity PNGs:", len(pngs))
print("Metrics:", metrics)
assert len(pngs) == 40
assert metrics.exists()
print("GATE 1: PASS")


In [ ]:
# Package Gate 1 outputs
archive_base = str(WORK / "HFSS_Gate1_results_2p45GHz")
zip_path = shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Created:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
